# Corpus samples: what did the filter keep, and what did it drop?

Reads every outlet's kept corpus (`data/<key>.csv`) into one DataFrame, then shows a random sample
of headlines with the first 100 words of each body for one outlet you pick. A second section does the
same for the articles the relevance filter dropped (`data/<key>_dropped.csv`, joined back to the
unfiltered file for the body text).

Requirements: pandas. Run top to bottom; change `OUTLET`, `N` and `SEED` and rerun the sample cells.

In [ ]:
import csv, sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, ".")
from outlets import OUTLETS as OUTLET_TABLE      # (key, label, group, logo, short)
csv.field_size_limit(10 ** 9)
DATA = Path("../data")
pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 200)

LABEL = {key: label for key, label, *_ in OUTLET_TABLE}
GROUP = {key: group for key, label, group, *_ in OUTLET_TABLE}

def first_words(text, n=100):
    words = str(text or "").split()
    return " ".join(words[:n]) + (" …" if len(words) > n else "")

## Every kept corpus in one DataFrame

In [ ]:
frames = []
for key, label, group, *_ in OUTLET_TABLE:
    path = DATA / f"{key}.csv"
    if not path.exists():
        print("missing:", path.name); continue
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    df["outlet"] = label; df["group"] = group; df["key"] = key
    frames.append(df)
kept = pd.concat(frames, ignore_index=True)
kept["date"] = kept["date"].str[:10]
kept["words"] = kept["body"].str.split().str.len()
print(f"{len(kept):,} kept articles across {kept['outlet'].nunique()} outlets")
(kept.groupby(["group", "outlet"], sort=False)
     .agg(articles=("url", "size"), with_text=("words", lambda w: int((w >= 80).sum())),
          median_words=("words", "median"), first=("date", "min"), last=("date", "max")))

## Random sample of kept articles

Pick an outlet by its label as the charts print it (`Fox News`, `ESPN`, `Washington Post`, `Breitbart`, …).
`SEED` fixes the draw so a sample can be shared; change it for a fresh one.

In [ ]:
OUTLET = "Fox News"
N = 100
SEED = 1

pool = kept[kept["outlet"] == OUTLET]
sample = pool.sample(n=min(N, len(pool)), random_state=SEED).sort_values("date")
print(f"{OUTLET}: {len(sample)} of {len(pool):,} kept articles")
view = sample.assign(opening=sample["body"].map(first_words))[["date", "headline", "opening", "url"]]
view.reset_index(drop=True)

## Random sample of dropped articles

The filter writes `data/<key>_dropped.csv` with date, headline, url and the reason. The body is not in
that file, so it is joined back from `data/<key>.unfiltered.csv` (the full harvest) by URL.

In [ ]:
def key_of(url):
    return str(url or "").split("?")[0].rstrip("/").lower()

drops = []
for key, label, group, *_ in OUTLET_TABLE:
    dpath, upath = DATA / f"{key}_dropped.csv", DATA / f"{key}.unfiltered.csv"
    if not dpath.exists():
        continue
    d = pd.read_csv(dpath, dtype=str, keep_default_na=False)
    if upath.exists():
        u = pd.read_csv(upath, dtype=str, keep_default_na=False, usecols=["url", "body"])
        u["k"] = u["url"].map(key_of)
        d["k"] = d["url"].map(key_of)
        d = d.merge(u.drop_duplicates("k")[["k", "body"]], on="k", how="left").drop(columns="k")
    else:
        d["body"] = ""
    d["outlet"] = label; d["group"] = group; d["key"] = key
    drops.append(d)
dropped = pd.concat(drops, ignore_index=True)
dropped["body"] = dropped["body"].fillna("")
print(f"{len(dropped):,} dropped articles")
dropped.groupby(["outlet", "reason"]).size().unstack(fill_value=0)

In [ ]:
OUTLET_DROPPED = "Fox News"
N_DROPPED = 100
SEED_DROPPED = 1

pool = dropped[dropped["outlet"] == OUTLET_DROPPED]
sample = pool.sample(n=min(N_DROPPED, len(pool)), random_state=SEED_DROPPED).sort_values("date")
print(f"{OUTLET_DROPPED}: {len(sample)} of {len(pool):,} dropped articles")
view = sample.assign(opening=sample["body"].map(first_words))[["date", "headline", "reason", "opening", "url"]]
view.reset_index(drop=True)

## Save a sample to share

Writes the current kept and dropped samples to `data/samples/` as CSV, so a collaborator can read them
without running anything.

In [ ]:
out = DATA / "samples"; out.mkdir(exist_ok=True)
k = kept[kept["outlet"] == OUTLET].sample(n=min(N, (kept["outlet"] == OUTLET).sum()), random_state=SEED)
k.assign(opening=k["body"].map(first_words))[["outlet", "date", "headline", "opening", "url"]].to_csv(out / f"kept_{OUTLET.lower().replace(' ', '_')}_{SEED}.csv", index=False)
dsel = dropped[dropped["outlet"] == OUTLET_DROPPED]
dsel = dsel.sample(n=min(N_DROPPED, len(dsel)), random_state=SEED_DROPPED)
dsel.assign(opening=dsel["body"].map(first_words))[["outlet", "date", "headline", "reason", "opening", "url"]].to_csv(out / f"dropped_{OUTLET_DROPPED.lower().replace(' ', '_')}_{SEED_DROPPED}.csv", index=False)
sorted(p.name for p in out.iterdir())